# CRM Data Quality & Campaign Readiness Dashboard — version corrigée

## Projet
**Objectif :** améliorer la qualité des données leads / contacts d’un CRM afin de fiabiliser les campagnes marketing et les décisions commerciales.

## Corrections apportées dans cette version

Au-delà du nettoyage des **numéros de téléphone** (normalisation, format
international `E.164`, détection des numéros trop courts / trop longs / vides /
suspects, gestion des extensions, librairie `phonenumbers`), cette version
corrige trois défauts de pipeline qui produisaient des KPI ininterprétables :

1. **Enrichissement contact via `account_id`.** Les leads ne contiennent pas de
   `contact_id` : le seul lien fiable vers la table `contacts` (qui porte les
   téléphones) est `converted_account_id` → `contacts.account_id`. La jointure
   se fait désormais sur ce lien, en retenant pour chaque compte le meilleur
   contact (téléphone valide prioritaire). Conséquence métier : le téléphone
   n'existe qu'après conversion (≈ 73 % des leads convertis ont un téléphone
   valide, contre ≈ 14 % sur l'ensemble).

2. **Détection de doublons corrigée.** L'ancienne logique marquait un doublon
   dès que le **nom d'entreprise** se répétait ; or le jeu synthétique ne compte
   qu'une vingtaine d'entreprises, ce qui marquait 100 % des leads comme
   doublons. Le doublon retenu (`is_duplicate`) repose maintenant sur un signal
   **fiable** : l'**email propre au lead** saisi plusieurs fois. Le couple
   nom+entreprise reste fourni à titre indicatif (`duplicate_company_name`).

3. **Jointure campagnes réparée.** Les `campaign_id` issus des activités
   étaient stockés en flottant (`"12.0"`) alors que la table `campaigns` utilise
   des entiers (`"12"`) : la jointure ne renvoyait aucune correspondance et
   `campaign_name` était vide. Les identifiants sont désormais normalisés en
   entiers-chaînes, ce qui rend la page **Qualité par campagne** exploitable.

> Le détail de ces choix est documenté dans `note_methodologique.md`.

## Dataset
Ce notebook utilise le **Bundle GTM relationnel** du projet `astronomer/mini-gtm-data-platform`, avec 4 fichiers principaux :

- `leads.csv`
- `contacts.csv`
- `campaigns.csv`
- `lead_activities.csv`

## Sorties finales pour Tableau

Le notebook produit :

- `data/processed/crm_quality_clean.csv` : table principale propre pour Tableau ;
- `data/processed/crm_quality_issues.csv` : journal des problèmes qualité détectés ;
- `data/processed/data_dictionary.csv` : dictionnaire des colonnes finales ;
- `reports/kpi_summary.csv` : résumé des KPIs principaux.

## Questions métier

1. Quels leads sont réellement exploitables pour une campagne marketing ?
2. Quelles sources / campagnes génèrent les données les plus fiables ?
3. Quels problèmes de qualité doivent être corrigés en priorité ?
4. Le CRM est-il assez fiable pour supporter le pilotage commercial ?

In [1]:
# ============================================================
# 1. Imports et configuration
# ============================================================

from pathlib import Path
from urllib.request import urlretrieve
from urllib.error import URLError, HTTPError
import re
import unicodedata
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 160)

RAW_DIR = Path("data/raw")
PROCESSED_DIR = Path("data/processed")
REPORTS_DIR = Path("reports")

for folder in [RAW_DIR, PROCESSED_DIR, REPORTS_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42

# Région par défaut pour parser les téléphones sans indicatif pays.
# Le bundle GTM contient généralement des données B2B orientées US.
# Change cette valeur en "FR" si tu veux traiter prioritairement des numéros français.
PHONE_DEFAULT_REGION = "US"

print("Environnement prêt.")
print("Dossier raw :", RAW_DIR.resolve())
print("Dossier processed :", PROCESSED_DIR.resolve())
print("Région par défaut pour les téléphones :", PHONE_DEFAULT_REGION)

Environnement prêt.
Dossier raw : C:\Users\user\Desktop\Study\M2\Data visualisations\Projet final\data\raw
Dossier processed : C:\Users\user\Desktop\Study\M2\Data visualisations\Projet final\data\processed
Région par défaut pour les téléphones : US


## 2. Téléchargement des données

Le code ci-dessous télécharge les 4 CSV bruts depuis GitHub.

Si le téléchargement échoue dans ton environnement, télécharge manuellement les fichiers suivants et place-les dans `data/raw/` avec ces noms exacts :

- `leads.csv`
- `contacts.csv`
- `campaigns.csv`
- `lead_activities.csv`

In [2]:
# ============================================================
# 2. Téléchargement des fichiers bruts
# ============================================================

DATA_URLS = {
    "leads": "https://raw.githubusercontent.com/astronomer/mini-gtm-data-platform/main/sources/postgres/leads.csv",
    "contacts": "https://raw.githubusercontent.com/astronomer/mini-gtm-data-platform/main/sources/postgres/contacts.csv",
    "campaigns": "https://raw.githubusercontent.com/astronomer/mini-gtm-data-platform/main/sources/postgres/campaigns.csv",
    "lead_activities": "https://raw.githubusercontent.com/astronomer/mini-gtm-data-platform/main/sources/postgres/lead_activities.csv",
}

def download_if_missing(name: str, url: str, raw_dir: Path = RAW_DIR) -> Path:
    output_file = raw_dir / f"{name}.csv"
    if output_file.exists():
        print(f"[OK] {output_file} existe déjà.")
        return output_file

    try:
        print(f"Téléchargement de {name}...")
        urlretrieve(url, output_file)
        print(f"[OK] Fichier téléchargé : {output_file}")
    except (HTTPError, URLError, Exception) as e:
        print(f"[ERREUR] Impossible de télécharger {name}.")
        print(f"URL : {url}")
        print(f"Détail : {e}")
        print(f"Action : télécharge le fichier manuellement et place-le ici : {output_file}")
    return output_file

raw_files = {name: download_if_missing(name, url) for name, url in DATA_URLS.items()}
raw_files

[OK] data\raw\leads.csv existe déjà.
[OK] data\raw\contacts.csv existe déjà.
[OK] data\raw\campaigns.csv existe déjà.
[OK] data\raw\lead_activities.csv existe déjà.


{'leads': WindowsPath('data/raw/leads.csv'),
 'contacts': WindowsPath('data/raw/contacts.csv'),
 'campaigns': WindowsPath('data/raw/campaigns.csv'),
 'lead_activities': WindowsPath('data/raw/lead_activities.csv')}

In [3]:
# ============================================================
# 3. Chargement des CSV
# ============================================================

def to_snake_case(col: str) -> str:
    col = str(col).strip()
    col = unicodedata.normalize("NFKD", col).encode("ascii", "ignore").decode("ascii")
    col = re.sub(r"[^0-9a-zA-Z]+", "_", col)
    col = re.sub(r"_+", "_", col)
    return col.strip("_").lower()

def read_csv_safe(path: Path) -> pd.DataFrame:
    if not path.exists():
        print(f"[WARN] Fichier absent : {path}")
        return pd.DataFrame()
    df = pd.read_csv(path)
    df.columns = [to_snake_case(c) for c in df.columns]
    return df

leads_raw = read_csv_safe(RAW_DIR / "leads.csv")
contacts_raw = read_csv_safe(RAW_DIR / "contacts.csv")
campaigns_raw = read_csv_safe(RAW_DIR / "campaigns.csv")
activities_raw = read_csv_safe(RAW_DIR / "lead_activities.csv")

tables = {
    "leads": leads_raw,
    "contacts": contacts_raw,
    "campaigns": campaigns_raw,
    "lead_activities": activities_raw,
}

for name, df in tables.items():
    print(f"\n--- {name} ---")
    print("Shape :", df.shape)
    print("Colonnes :", list(df.columns))
    display(df.head(3))


--- leads ---
Shape : (8000, 12)
Colonnes : ['lead_id', 'first_name', 'last_name', 'email', 'company', 'source', 'status', 'score', 'converted_opp_id', 'converted_account_id', 'converted_date', 'created_at']


,lead_id,first_name,last_name,email,company,source,status,score,converted_opp_id,converted_account_id,converted_date,created_at
0,1,Sarah,Martinez,sarah.martinez1@insightplatforms.com,Insight Platforms,Social Media,MQL,64,NaN,NaN,NaN,2021-05-27T00:00:00
1,2,Sarah,Garcia,sarah.garcia2@techflowinc.com,TechFlow Inc,Product-Led,Working,31,NaN,NaN,NaN,2020-01-17T00:00:00
2,3,Robert,Lee,robert.lee3@agiledynamics.com,Agile Dynamics,Event,Converted,66,924.0,303.0,2024-11-30,2024-09-23T00:00:00



--- contacts ---
Shape : (3000, 9)
Colonnes : ['contact_id', 'account_id', 'first_name', 'last_name', 'email', 'phone', 'title', 'role', 'created_at']


,contact_id,account_id,first_name,last_name,email,phone,title,role,created_at
0,1,466,Betty,Johnson,betty.johnson@piedpiper.com,+1-948-790-1355,Chief Revenue Officer,Champion,2021-08-08T00:00:00
1,2,507,Steven,Davis,steven.davis,+1-763-893-9070,Director of IT,Technical Evaluator,2022-10-03T00:00:00
2,3,609,Kimberly,Perez,kimberly.perez@acme.com,+1-807-884-2849,IT Manager,Blocker,2024-11-02T00:00:00



--- campaigns ---
Shape : (200, 9)
Colonnes : ['campaign_id', 'name', 'channel', 'type', 'status', 'cost', 'start_date', 'end_date', 'created_at']


,campaign_id,name,channel,type,status,cost,start_date,end_date,created_at
0,1,Paid Search - Event Promotion - Q4 2022,Paid Search,Event Promotion,Paused,54507.0,2022-10-07,2022-12-18,2022-09-23T00:00:00
1,2,Organic Search - Brand Awareness - Q4 2021,Organic Search,Brand Awareness,Completed,2955.0,2021-12-20,2022-02-19,2021-12-09T00:00:00
2,3,Webinar - Thought Leadership - Q2 2024,Webinar,Thought Leadership,Active,26974.0,2024-04-25,2024-07-11,2024-04-19T00:00:00



--- lead_activities ---
Shape : (30000, 6)
Colonnes : ['activity_id', 'lead_id', 'campaign_id', 'activity_type', 'activity_date', 'utm_source']


,activity_id,lead_id,campaign_id,activity_type,activity_date,utm_source
0,1,6719,12.0,Unsubscribe,2021-09-07T20:57:00,NaN
1,2,6538,75.0,Email Open,2023-08-26T05:45:00,NaN
2,3,2137,95.0,Webinar Attended,2021-03-25T05:26:00,NaN


## 4. Profilage initial

Cette étape documente l’état initial de la qualité des données avant nettoyage :

- nombre de lignes ;
- nombre de colonnes ;
- taux de valeurs manquantes ;
- doublons exacts ;
- types de données.

In [4]:
# ============================================================
# 4. Profilage initial
# ============================================================

def profile_table(df: pd.DataFrame, table_name: str) -> pd.DataFrame:
    if df.empty:
        return pd.DataFrame([{
            "table": table_name,
            "rows": 0,
            "columns": 0,
            "duplicate_rows": 0,
            "missing_cells": 0,
            "missing_cells_pct": np.nan
        }])
    total_cells = df.shape[0] * df.shape[1]
    return pd.DataFrame([{
        "table": table_name,
        "rows": df.shape[0],
        "columns": df.shape[1],
        "duplicate_rows": int(df.duplicated().sum()),
        "missing_cells": int(df.isna().sum().sum()),
        "missing_cells_pct": round(df.isna().sum().sum() / total_cells * 100, 2) if total_cells else 0
    }])

profile_summary = pd.concat(
    [profile_table(df, name) for name, df in tables.items()],
    ignore_index=True
)

display(profile_summary)

missing_details = []
for table_name, df in tables.items():
    if df.empty:
        continue
    tmp = (
        df.isna().mean()
        .mul(100)
        .round(2)
        .reset_index()
        .rename(columns={"index": "column", 0: "missing_pct"})
    )
    tmp["table"] = table_name
    tmp["non_null_count"] = df.notna().sum().values
    tmp["dtype"] = df.dtypes.astype(str).values
    missing_details.append(tmp)

missing_details = pd.concat(missing_details, ignore_index=True) if missing_details else pd.DataFrame()
display(missing_details.sort_values(["missing_pct", "table"], ascending=[False, True]).head(30))

,table,rows,columns,duplicate_rows,missing_cells,missing_cells_pct
0,leads,8000,12,0,19932,20.76
1,contacts,3000,9,0,364,1.35
2,campaigns,200,9,0,11,0.61
3,lead_activities,30000,6,0,23590,13.11


,column,missing_pct,table,non_null_count,dtype
8,converted_opp_id,80.73,leads,1542,float64
9,converted_account_id,80.73,leads,1542,float64
10,converted_date,80.73,leads,1542,str
35,utm_source,71.75,lead_activities,8476,str
17,phone,7.50,contacts,2775,str
26,cost,5.00,campaigns,190,float64
32,campaign_id,4.04,lead_activities,28788,float64
3,email,3.92,leads,7686,str
5,source,3.05,leads,7756,str
33,activity_type,2.85,lead_activities,29146,str


## 5. Fonctions utilitaires de nettoyage

Cette version corrige la gestion des numéros de téléphone.

La fonction `clean_phone()` retourne plusieurs informations :

- `phone_clean` : numéro propre au format international `E.164` quand il est valide ;
- `phone_digits` : chiffres extraits ;
- `phone_valid` : booléen de validité ;
- `phone_issue` : raison d’erreur ou `OK` ;
- `phone_extension` : extension éventuelle.

Le notebook utilise la librairie `phonenumbers` si elle est installée. Sinon, il applique une logique de secours robuste.

In [5]:
# ============================================================
# 5. Fonctions utilitaires
# ============================================================

EMAIL_RE = re.compile(r"^[A-Za-z0-9._%+\-]+@[A-Za-z0-9.\-]+\.[A-Za-z]{2,}$")

try:
    import phonenumbers
    from phonenumbers import NumberParseException, PhoneNumberFormat
    HAS_PHONENUMBERS = True
except Exception:
    phonenumbers = None
    NumberParseException = Exception
    PhoneNumberFormat = None
    HAS_PHONENUMBERS = False

print("Librairie phonenumbers disponible :", HAS_PHONENUMBERS)

def normalize_text(x, empty_as_na=True):
    """Nettoie un texte : trim, espaces multiples, unicode."""
    if pd.isna(x):
        return pd.NA if empty_as_na else ""
    s = str(x).strip()
    s = re.sub(r"\s+", " ", s)
    if s == "":
        return pd.NA if empty_as_na else ""
    return s

def normalize_text_key(x):
    """Version standardisée pour comparaisons, dédoublonnage et mapping."""
    if pd.isna(x):
        return ""
    s = str(x).strip().lower()
    s = unicodedata.normalize("NFKD", s).encode("ascii", "ignore").decode("ascii")
    s = re.sub(r"[^a-z0-9]+", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def normalize_email(x):
    if pd.isna(x):
        return pd.NA
    s = str(x).strip().lower()
    s = re.sub(r"\s+", "", s)
    return s if s else pd.NA

def is_valid_email(x):
    if pd.isna(x):
        return False
    return bool(EMAIL_RE.match(str(x)))

def _extract_phone_extension(raw: str):
    """Sépare le numéro principal et l'extension éventuelle."""
    if raw is None or pd.isna(raw):
        return "", pd.NA

    s = str(raw).strip()
    if not s:
        return "", pd.NA

    # Exemples capturés : "ext 123", "ext.123", "x123", "#123"
    ext_match = re.search(r"(?:ext\.?|extension|x|#)\s*(\d{1,8})\s*$", s, flags=re.IGNORECASE)
    extension = pd.NA
    if ext_match:
        extension = ext_match.group(1)
        s = s[:ext_match.start()].strip()

    return s, extension

def clean_phone(raw_phone, default_region=PHONE_DEFAULT_REGION):
    """
    Nettoie et valide un numéro de téléphone.

    Retourne un dictionnaire :
    - phone_clean : numéro au format E.164 si valide, sinon NA ;
    - phone_digits : chiffres extraits ;
    - phone_valid : booléen ;
    - phone_issue : raison de l'erreur ou OK ;
    - phone_extension : extension éventuelle ;
    - phone_clean_display : version lisible même si non valide.
    """
    if pd.isna(raw_phone):
        return {
            "phone_clean": pd.NA,
            "phone_digits": pd.NA,
            "phone_valid": False,
            "phone_issue": "Téléphone manquant",
            "phone_extension": pd.NA,
            "phone_clean_display": pd.NA,
        }

    original = str(raw_phone).strip()
    if original == "" or original.lower() in {"nan", "none", "null", "n/a", "na"}:
        return {
            "phone_clean": pd.NA,
            "phone_digits": pd.NA,
            "phone_valid": False,
            "phone_issue": "Téléphone manquant",
            "phone_extension": pd.NA,
            "phone_clean_display": pd.NA,
        }

    main_phone, extension = _extract_phone_extension(original)

    # Conversion de préfixe international 00 vers +
    main_phone = re.sub(r"^\s*00", "+", main_phone)

    # Suppression des caractères de présentation en conservant le + initial
    cleaned = main_phone.strip()
    cleaned = re.sub(r"[^\d+]", "", cleaned)
    if cleaned.count("+") > 1:
        cleaned = "+" + cleaned.replace("+", "")
    elif "+" in cleaned and not cleaned.startswith("+"):
        cleaned = "+" + cleaned.replace("+", "")

    digits = re.sub(r"\D", "", cleaned)
    display_value = cleaned if cleaned else pd.NA

    if digits == "":
        return {
            "phone_clean": pd.NA,
            "phone_digits": pd.NA,
            "phone_valid": False,
            "phone_issue": "Aucun chiffre détecté",
            "phone_extension": extension,
            "phone_clean_display": display_value,
        }

    if len(set(digits)) == 1 and len(digits) >= 6:
        return {
            "phone_clean": pd.NA,
            "phone_digits": digits,
            "phone_valid": False,
            "phone_issue": "Numéro suspect : chiffre répété",
            "phone_extension": extension,
            "phone_clean_display": display_value,
        }

    if HAS_PHONENUMBERS:
        try:
            # Si le numéro commence par +, la région n'est pas nécessaire.
            parsed = phonenumbers.parse(cleaned, None if cleaned.startswith("+") else default_region)

            if phonenumbers.is_valid_number(parsed):
                e164 = phonenumbers.format_number(parsed, PhoneNumberFormat.E164)
                return {
                    "phone_clean": e164,
                    "phone_digits": re.sub(r"\D", "", e164),
                    "phone_valid": True,
                    "phone_issue": "OK",
                    "phone_extension": extension,
                    "phone_clean_display": e164,
                }

            if phonenumbers.is_possible_number(parsed):
                return {
                    "phone_clean": pd.NA,
                    "phone_digits": digits,
                    "phone_valid": False,
                    "phone_issue": "Format possible mais numéro non validé",
                    "phone_extension": extension,
                    "phone_clean_display": display_value,
                }

        except NumberParseException:
            pass

    # Fallback robuste si phonenumbers est absent ou si le parsing échoue.
    if len(digits) < 8:
        issue = "Téléphone trop court"
        valid = False
        e164 = pd.NA
    elif len(digits) > 15:
        issue = "Téléphone trop long"
        valid = False
        e164 = pd.NA
    else:
        # Cas US par défaut : 10 chiffres => +1XXXXXXXXXX
        if default_region.upper() == "US" and len(digits) == 10:
            e164 = "+1" + digits
            valid = True
            issue = "OK"
        elif default_region.upper() == "US" and len(digits) == 11 and digits.startswith("1"):
            e164 = "+" + digits
            valid = True
            issue = "OK"
        # Cas France simple : 10 chiffres commençant par 0 => +33...
        elif default_region.upper() == "FR" and len(digits) == 10 and digits.startswith("0"):
            e164 = "+33" + digits[1:]
            valid = True
            issue = "OK"
        elif cleaned.startswith("+") and 8 <= len(digits) <= 15:
            e164 = "+" + digits
            valid = True
            issue = "OK"
        else:
            e164 = pd.NA
            valid = False
            issue = "Indicatif pays manquant ou format ambigu"

    return {
        "phone_clean": e164,
        "phone_digits": digits,
        "phone_valid": bool(valid),
        "phone_issue": issue,
        "phone_extension": extension,
        "phone_clean_display": e164 if valid else display_value,
    }

def phone_info_dataframe(series: pd.Series, prefix: str, default_region=PHONE_DEFAULT_REGION) -> pd.DataFrame:
    """Applique clean_phone() et préfixe les colonnes produites."""
    info = series.apply(lambda x: pd.Series(clean_phone(x, default_region=default_region)))
    return info.add_prefix(prefix)

def parse_date_safe(s):
    return pd.to_datetime(s, errors="coerce", utc=False)

def find_col(df, candidates, required=False, table_name=""):
    """
    Retourne le premier nom de colonne existant dans df parmi une liste de candidats.
    Fonctionne avec des noms déjà passés en snake_case.
    """
    cols = set(df.columns)
    for cand in candidates:
        cand_snake = to_snake_case(cand)
        if cand_snake in cols:
            return cand_snake
    if required:
        raise KeyError(
            f"Colonne obligatoire introuvable dans {table_name}. "
            f"Candidats : {candidates}. Colonnes disponibles : {list(df.columns)}"
        )
    return None

def quality_level(score):
    if pd.isna(score):
        return "Non calculé"
    if score >= 80:
        return "Excellent"
    if score >= 60:
        return "Correct"
    if score >= 40:
        return "À corriger"
    return "Critique"

def yes_no(flag):
    return np.where(flag, "Oui", "Non")

Librairie phonenumbers disponible : True


## 6. Nettoyage de la table `contacts`

Objectifs :

- normaliser les emails ;
- normaliser les téléphones ;
- créer des flags de validité ;
- préparer la jointure avec les leads.

In [6]:
# ============================================================
# 6. Nettoyage contacts
# ============================================================

contacts = contacts_raw.copy()

if contacts.empty:
    print("[WARN] contacts.csv absent ou vide.")
    contacts_clean = pd.DataFrame()
else:
    contact_id_col = find_col(contacts, ["contact_id", "id"], required=False, table_name="contacts")
    account_id_col = find_col(contacts, ["account_id", "account"], required=False, table_name="contacts")
    role_col = find_col(contacts, ["role", "persona"], required=False, table_name="contacts")
    email_col = find_col(contacts, ["email", "email_address", "work_email"], required=False, table_name="contacts")
    phone_col = find_col(contacts, ["phone", "phone_number", "mobile", "mobile_phone"], required=False, table_name="contacts")
    first_name_col = find_col(contacts, ["first_name", "firstname", "first"], required=False, table_name="contacts")
    last_name_col = find_col(contacts, ["last_name", "lastname", "last"], required=False, table_name="contacts")
    company_col = find_col(contacts, ["company", "company_name", "account_name", "organization"], required=False, table_name="contacts")

    if contact_id_col is None:
        contacts["contact_id"] = np.arange(1, len(contacts) + 1)
        contact_id_col = "contact_id"

    contacts_clean = pd.DataFrame()
    contacts_clean["contact_id"] = contacts[contact_id_col].astype(str)

    # account_id : cle de rattachement aux leads convertis (converted_account_id).
    if account_id_col:
        acc = pd.to_numeric(contacts[account_id_col], errors="coerce").astype("Int64")
        contacts_clean["account_id"] = acc.astype(str).replace("<NA>", pd.NA)
    else:
        contacts_clean["account_id"] = pd.NA

    contacts_clean["contact_role"] = contacts[role_col].map(normalize_text) if role_col else pd.NA

    contacts_clean["contact_first_name"] = contacts[first_name_col].map(normalize_text) if first_name_col else pd.NA
    contacts_clean["contact_last_name"] = contacts[last_name_col].map(normalize_text) if last_name_col else pd.NA
    contacts_clean["contact_company"] = contacts[company_col].map(normalize_text) if company_col else pd.NA

    contacts_clean["contact_email_raw"] = contacts[email_col] if email_col else pd.NA
    contacts_clean["contact_email_clean"] = contacts_clean["contact_email_raw"].map(normalize_email)
    contacts_clean["contact_email_valid"] = contacts_clean["contact_email_clean"].map(is_valid_email)

    contacts_clean["contact_phone_raw"] = contacts[phone_col] if phone_col else pd.NA
    contact_phone_info = phone_info_dataframe(
        contacts_clean["contact_phone_raw"],
        prefix="contact_",
        default_region=PHONE_DEFAULT_REGION
    )
    contacts_clean = pd.concat([contacts_clean, contact_phone_info], axis=1)

    contacts_clean["contact_name_key"] = (
        contacts_clean["contact_first_name"].map(normalize_text_key).fillna("") + " " +
        contacts_clean["contact_last_name"].map(normalize_text_key).fillna("")
    ).str.strip()

    contacts_clean["contact_company_key"] = contacts_clean["contact_company"].map(normalize_text_key)

    print("Colonnes detectees contacts :")
    print({
        "contact_id_col": contact_id_col,
        "account_id_col": account_id_col,
        "role_col": role_col,
        "email_col": email_col,
        "phone_col": phone_col,
        "first_name_col": first_name_col,
        "last_name_col": last_name_col,
        "company_col": company_col,
    })

    display(contacts_clean.head())

    contact_quality_summary = pd.DataFrame({
        "metric": ["Emails valides (%)", "Telephones valides (%)"],
        "value": [
            round(contacts_clean["contact_email_valid"].mean() * 100, 2),
            round(contacts_clean["contact_phone_valid"].mean() * 100, 2)
        ]
    })
    display(contact_quality_summary)

    print("Repartition des erreurs telephone contacts :")
    display(contacts_clean["contact_phone_issue"].value_counts(dropna=False).rename_axis("phone_issue").reset_index(name="count").head(20))


Colonnes detectees contacts :
{'contact_id_col': 'contact_id', 'account_id_col': 'account_id', 'role_col': 'role', 'email_col': 'email', 'phone_col': 'phone', 'first_name_col': 'first_name', 'last_name_col': 'last_name', 'company_col': None}


,contact_id,account_id,contact_role,contact_first_name,contact_last_name,contact_company,contact_email_raw,contact_email_clean,contact_email_valid,contact_phone_raw,contact_phone_clean,contact_phone_digits,contact_phone_valid,contact_phone_issue,contact_phone_extension,contact_phone_clean_display,contact_name_key,contact_company_key
0,1,466,Champion,Betty,Johnson,<NA>,betty.johnson@piedpiper.com,betty.johnson@piedpiper.com,True,+1-948-790-1355,+19487901355,19487901355,True,OK,<NA>,+19487901355,betty johnson,
1,2,507,Technical Evaluator,Steven,Davis,<NA>,steven.davis,steven.davis,False,+1-763-893-9070,+17638939070,17638939070,True,OK,<NA>,+17638939070,steven davis,
2,3,609,Blocker,Kimberly,Perez,<NA>,kimberly.perez@acme.com,kimberly.perez@acme.com,True,+1-807-884-2849,+18078842849,18078842849,True,OK,<NA>,+18078842849,kimberly perez,
3,4,760,Champion,Sandra,Jackson,<NA>,sandra.jackson@globex.com,sandra.jackson@globex.com,True,+1-882-865-1156,NaN,18828651156,False,Format possible mais numéro non validé,<NA>,+18828651156,sandra jackson,
4,5,520,Technical Evaluator,David,Walker,<NA>,david.walker@piedpiper.com,david.walker@piedpiper.com,True,+1-712-624-5095,+17126245095,17126245095,True,OK,<NA>,+17126245095,david walker,


,metric,value
0,Emails valides (%),92.03
1,Telephones valides (%),45.13


Repartition des erreurs telephone contacts :


,phone_issue,count
0,Format possible mais numéro non validé,1421
1,OK,1354
2,Téléphone manquant,225


## 7. Nettoyage de la table `leads`

Objectifs :

- préparer la table principale ;
- normaliser les sources ;
- parser les dates ;
- récupérer les emails / téléphones si présents dans `leads`.

In [7]:
# ============================================================
# 7. Nettoyage leads
# ============================================================

leads = leads_raw.copy()

if leads.empty:
    raise ValueError("leads.csv est obligatoire pour ce projet.")

lead_id_col = find_col(leads, ["lead_id", "id"], required=False, table_name="leads")
contact_fk_col = find_col(leads, ["contact_id", "person_id"], required=False, table_name="leads")
account_fk_col = find_col(leads, ["converted_account_id", "account_id"], required=False, table_name="leads")
campaign_fk_col = find_col(leads, ["campaign_id"], required=False, table_name="leads")

source_col = find_col(leads, ["source", "lead_source", "utm_source", "channel"], required=False, table_name="leads")
status_col = find_col(leads, ["status", "lead_status", "stage", "deal_stage", "lifecycle_stage"], required=False, table_name="leads")
created_col = find_col(leads, ["created_at", "created_date", "created_on", "date_created"], required=False, table_name="leads")
converted_col = find_col(leads, ["converted_date", "converted_at"], required=False, table_name="leads")

lead_email_col = find_col(leads, ["email", "email_address"], required=False, table_name="leads")
lead_phone_col = find_col(leads, ["phone", "phone_number", "mobile"], required=False, table_name="leads")
lead_company_col = find_col(leads, ["company", "company_name", "account_name"], required=False, table_name="leads")
lead_owner_col = find_col(leads, ["lead_owner", "owner", "sales_owner", "assigned_to"], required=False, table_name="leads")
lead_first_name_col = find_col(leads, ["first_name", "firstname", "first"], required=False, table_name="leads")
lead_last_name_col = find_col(leads, ["last_name", "lastname", "last"], required=False, table_name="leads")

if lead_id_col is None:
    leads["lead_id"] = np.arange(1, len(leads) + 1)
    lead_id_col = "lead_id"

leads_clean = pd.DataFrame()
leads_clean["lead_id"] = leads[lead_id_col].astype(str)

if contact_fk_col:
    leads_clean["contact_id"] = leads[contact_fk_col].astype(str)
else:
    leads_clean["contact_id"] = pd.NA

# account_id : seul lien fiable vers la table contacts (via converted_account_id).
if account_fk_col:
    acc = pd.to_numeric(leads[account_fk_col], errors="coerce").astype("Int64")
    leads_clean["account_id"] = acc.astype(str).replace("<NA>", pd.NA)
else:
    leads_clean["account_id"] = pd.NA

if campaign_fk_col:
    leads_clean["lead_campaign_id"] = leads[campaign_fk_col].astype(str)
else:
    leads_clean["lead_campaign_id"] = pd.NA

leads_clean["lead_source_raw"] = leads[source_col] if source_col else pd.NA
leads_clean["source_clean"] = leads_clean["lead_source_raw"].map(normalize_text_key).replace("", pd.NA)

leads_clean["lead_status_raw"] = leads[status_col] if status_col else pd.NA
leads_clean["lead_status_clean"] = leads_clean["lead_status_raw"].map(normalize_text_key).replace("", pd.NA)

leads_clean["lead_created_at"] = parse_date_safe(leads[created_col]) if created_col else pd.NaT
leads_clean["lead_converted_at"] = parse_date_safe(leads[converted_col]) if converted_col else pd.NaT

leads_clean["lead_email_raw"] = leads[lead_email_col] if lead_email_col else pd.NA
leads_clean["lead_email_clean"] = leads_clean["lead_email_raw"].map(normalize_email)
leads_clean["lead_email_valid"] = leads_clean["lead_email_clean"].map(is_valid_email)

leads_clean["lead_phone_raw"] = leads[lead_phone_col] if lead_phone_col else pd.NA
lead_phone_info = phone_info_dataframe(
    leads_clean["lead_phone_raw"],
    prefix="lead_",
    default_region=PHONE_DEFAULT_REGION
)
leads_clean = pd.concat([leads_clean, lead_phone_info], axis=1)

leads_clean["lead_company"] = leads[lead_company_col].map(normalize_text) if lead_company_col else pd.NA
leads_clean["lead_company_key"] = leads_clean["lead_company"].map(normalize_text_key)

# Nom du lead : utilise pour la detection de doublons "personne".
leads_clean["lead_first_name"] = leads[lead_first_name_col].map(normalize_text) if lead_first_name_col else pd.NA
leads_clean["lead_last_name"] = leads[lead_last_name_col].map(normalize_text) if lead_last_name_col else pd.NA
leads_clean["lead_name_key"] = (
    leads_clean["lead_first_name"].map(normalize_text_key).fillna("") + " " +
    leads_clean["lead_last_name"].map(normalize_text_key).fillna("")
).str.strip()

leads_clean["lead_owner"] = leads[lead_owner_col].map(normalize_text) if lead_owner_col else "Non renseigne"

print("Colonnes detectees leads :")
print({
    "lead_id_col": lead_id_col,
    "contact_fk_col": contact_fk_col,
    "account_fk_col": account_fk_col,
    "campaign_fk_col": campaign_fk_col,
    "source_col": source_col,
    "status_col": status_col,
    "created_col": created_col,
    "converted_col": converted_col,
    "lead_email_col": lead_email_col,
    "lead_phone_col": lead_phone_col,
    "lead_company_col": lead_company_col,
    "lead_owner_col": lead_owner_col,
})

display(leads_clean.head())

print("Repartition des erreurs telephone leads :")
display(leads_clean["lead_phone_issue"].value_counts(dropna=False).rename_axis("phone_issue").reset_index(name="count").head(20))


Colonnes detectees leads :
{'lead_id_col': 'lead_id', 'contact_fk_col': None, 'account_fk_col': 'converted_account_id', 'campaign_fk_col': None, 'source_col': 'source', 'status_col': 'status', 'created_col': 'created_at', 'converted_col': 'converted_date', 'lead_email_col': 'email', 'lead_phone_col': None, 'lead_company_col': 'company', 'lead_owner_col': None}


,lead_id,contact_id,account_id,lead_campaign_id,lead_source_raw,source_clean,lead_status_raw,lead_status_clean,lead_created_at,lead_converted_at,lead_email_raw,lead_email_clean,lead_email_valid,lead_phone_raw,lead_phone_clean,lead_phone_digits,lead_phone_valid,lead_phone_issue,lead_phone_extension,lead_phone_clean_display,lead_company,lead_company_key,lead_first_name,lead_last_name,lead_name_key,lead_owner
0,1,<NA>,NaN,<NA>,Social Media,social media,MQL,mql,2021-05-27,NaT,sarah.martinez1@insightplatforms.com,sarah.martinez1@insightplatforms.com,True,<NA>,<NA>,<NA>,False,Téléphone manquant,<NA>,<NA>,Insight Platforms,insight platforms,Sarah,Martinez,sarah martinez,Non renseigne
1,2,<NA>,NaN,<NA>,Product-Led,product led,Working,working,2020-01-17,NaT,sarah.garcia2@techflowinc.com,sarah.garcia2@techflowinc.com,True,<NA>,<NA>,<NA>,False,Téléphone manquant,<NA>,<NA>,TechFlow Inc,techflow inc,Sarah,Garcia,sarah garcia,Non renseigne
2,3,<NA>,303,<NA>,Event,event,Converted,converted,2024-09-23,2024-11-30,robert.lee3@agiledynamics.com,robert.lee3@agiledynamics.com,True,<NA>,<NA>,<NA>,False,Téléphone manquant,<NA>,<NA>,Agile Dynamics,agile dynamics,Robert,Lee,robert lee,Non renseigne
3,4,<NA>,NaN,<NA>,Outbound - Cold,outbound cold,MQL,mql,2022-05-29,NaT,daniel.rodriguez4@metasync.com,daniel.rodriguez4@metasync.com,True,<NA>,<NA>,<NA>,False,Téléphone manquant,<NA>,<NA>,MetaSync,metasync,Daniel,Rodriguez,daniel rodriguez,Non renseigne
4,5,<NA>,NaN,<NA>,Inbound - Webinar,inbound webinar,SQL,sql,2024-07-27,NaT,karen.lee5@scaleupsystems.com,karen.lee5@scaleupsystems.com,True,<NA>,<NA>,<NA>,False,Téléphone manquant,<NA>,<NA>,ScaleUp Systems,scaleup systems,Karen,Lee,karen lee,Non renseigne


Repartition des erreurs telephone leads :


,phone_issue,count
0,Téléphone manquant,8000


## 8. Nettoyage de la table `campaigns`

Objectifs :

- normaliser les noms de campagnes ;
- parser les dates de début / fin ;
- normaliser les canaux ou sources marketing.

In [8]:
# ============================================================
# 8. Nettoyage campaigns
# ============================================================

campaigns = campaigns_raw.copy()

if campaigns.empty:
    print("[WARN] campaigns.csv absent ou vide.")
    campaigns_clean = pd.DataFrame(columns=[
        "campaign_id", "campaign_name", "campaign_channel", "campaign_source",
        "campaign_start_date", "campaign_end_date"
    ])
else:
    campaign_id_col = find_col(campaigns, ["campaign_id", "id"], required=False, table_name="campaigns")
    campaign_name_col = find_col(campaigns, ["campaign_name", "name"], required=False, table_name="campaigns")
    channel_col = find_col(campaigns, ["channel", "campaign_channel", "type"], required=False, table_name="campaigns")
    campaign_source_col = find_col(campaigns, ["source", "utm_source"], required=False, table_name="campaigns")
    start_col = find_col(campaigns, ["start_date", "started_at", "start_at"], required=False, table_name="campaigns")
    end_col = find_col(campaigns, ["end_date", "ended_at", "end_at"], required=False, table_name="campaigns")

    if campaign_id_col is None:
        campaigns["campaign_id"] = np.arange(1, len(campaigns) + 1)
        campaign_id_col = "campaign_id"

    campaigns_clean = pd.DataFrame()
    campaigns_clean["campaign_id"] = (
        pd.to_numeric(campaigns[campaign_id_col], errors="coerce")
        .astype("Int64").astype("string")
    )
    campaigns_clean["campaign_name"] = campaigns[campaign_name_col].map(normalize_text) if campaign_name_col else pd.NA
    campaigns_clean["campaign_channel"] = campaigns[channel_col].map(normalize_text_key).replace("", pd.NA) if channel_col else pd.NA
    campaigns_clean["campaign_source"] = campaigns[campaign_source_col].map(normalize_text_key).replace("", pd.NA) if campaign_source_col else campaigns_clean["campaign_channel"]
    campaigns_clean["campaign_start_date"] = parse_date_safe(campaigns[start_col]) if start_col else pd.NaT
    campaigns_clean["campaign_end_date"] = parse_date_safe(campaigns[end_col]) if end_col else pd.NaT

    print("Colonnes détectées campaigns :")
    print({
        "campaign_id_col": campaign_id_col,
        "campaign_name_col": campaign_name_col,
        "channel_col": channel_col,
        "campaign_source_col": campaign_source_col,
        "start_col": start_col,
        "end_col": end_col,
    })
    display(campaigns_clean.head())

Colonnes détectées campaigns :
{'campaign_id_col': 'campaign_id', 'campaign_name_col': 'name', 'channel_col': 'channel', 'campaign_source_col': None, 'start_col': 'start_date', 'end_col': 'end_date'}


,campaign_id,campaign_name,campaign_channel,campaign_source,campaign_start_date,campaign_end_date
0,1,Paid Search - Event Promotion - Q4 2022,paid search,paid search,2022-10-07,2022-12-18
1,2,Organic Search - Brand Awareness - Q4 2021,organic search,organic search,2021-12-20,2022-02-19
2,3,Webinar - Thought Leadership - Q2 2024,webinar,webinar,2024-04-25,2024-07-11
3,4,Webinar - Demand Gen - Q2 2024,webinar,webinar,2024-06-28,2024-10-12
4,5,FY2023 Brand Awareness - Organic Search,organic search,organic search,2023-05-23,2023-06-14


## 9. Nettoyage de la table `lead_activities`

Objectifs :

- parser les dates d’activités ;
- identifier le type d’activité ;
- agréger les activités par lead ;
- produire des indicateurs utiles pour Tableau.

In [9]:
# ============================================================
# 9. Nettoyage lead_activities
# ============================================================

activities = activities_raw.copy()

if activities.empty:
    print("[WARN] lead_activities.csv absent ou vide.")
    activities_clean = pd.DataFrame()
    activities_by_lead = pd.DataFrame(columns=[
        "lead_id", "total_activities", "first_activity_date", "last_activity_date",
        "unique_campaigns_count", "activity_types_count", "has_activity"
    ])
else:
    activity_id_col = find_col(activities, ["activity_id", "id"], required=False, table_name="lead_activities")
    act_lead_id_col = find_col(activities, ["lead_id"], required=True, table_name="lead_activities")
    act_campaign_id_col = find_col(activities, ["campaign_id"], required=False, table_name="lead_activities")
    activity_type_col = find_col(activities, ["activity_type", "type", "event_type", "event"], required=False, table_name="lead_activities")
    activity_date_col = find_col(activities, ["activity_date", "created_at", "event_date", "timestamp", "occurred_at"], required=False, table_name="lead_activities")

    if activity_id_col is None:
        activities["activity_id"] = np.arange(1, len(activities) + 1)
        activity_id_col = "activity_id"

    activities_clean = pd.DataFrame()
    activities_clean["activity_id"] = activities[activity_id_col].astype(str)
    activities_clean["lead_id"] = activities[act_lead_id_col].astype(str)
    activities_clean["campaign_id"] = (
        pd.to_numeric(activities[act_campaign_id_col], errors="coerce")
        .astype("Int64").astype("string")
    ) if act_campaign_id_col else pd.NA
    activities_clean["activity_type"] = activities[activity_type_col].map(normalize_text_key).replace("", pd.NA) if activity_type_col else pd.NA
    activities_clean["activity_date"] = parse_date_safe(activities[activity_date_col]) if activity_date_col else pd.NaT

    activities_clean["activity_date_valid"] = activities_clean["activity_date"].notna()
    today = pd.Timestamp.today().normalize()
    activities_clean["activity_future_date"] = activities_clean["activity_date"].dt.tz_localize(None) > today

    activities_by_lead = (
        activities_clean
        .groupby("lead_id", dropna=False)
        .agg(
            total_activities=("activity_id", "count"),
            first_activity_date=("activity_date", "min"),
            last_activity_date=("activity_date", "max"),
            unique_campaigns_count=("campaign_id", lambda x: x.dropna().nunique()),
            activity_types_count=("activity_type", lambda x: x.dropna().nunique()),
            invalid_activity_dates=("activity_date_valid", lambda x: int((~x).sum())),
            future_activity_dates=("activity_future_date", "sum")
        )
        .reset_index()
    )
    activities_by_lead["has_activity"] = activities_by_lead["total_activities"] > 0

    print("Colonnes détectées lead_activities :")
    print({
        "activity_id_col": activity_id_col,
        "act_lead_id_col": act_lead_id_col,
        "act_campaign_id_col": act_campaign_id_col,
        "activity_type_col": activity_type_col,
        "activity_date_col": activity_date_col,
    })

    display(activities_clean.head())
    display(activities_by_lead.head())

Colonnes détectées lead_activities :
{'activity_id_col': 'activity_id', 'act_lead_id_col': 'lead_id', 'act_campaign_id_col': 'campaign_id', 'activity_type_col': 'activity_type', 'activity_date_col': 'activity_date'}


,activity_id,lead_id,campaign_id,activity_type,activity_date,activity_date_valid,activity_future_date
0,1,6719,12,unsubscribe,2021-09-07 20:57:00,True,False
1,2,6538,75,email open,2023-08-26 05:45:00,True,False
2,3,2137,95,webinar attended,2021-03-25 05:26:00,True,False
3,4,2315,52,ad click,2024-02-07 18:34:00,True,False
4,5,7042,192,email click,2020-12-01 13:42:00,True,False


,lead_id,total_activities,first_activity_date,last_activity_date,unique_campaigns_count,activity_types_count,invalid_activity_dates,future_activity_dates,has_activity
0,1,4,2020-12-04 21:08:00,2024-05-09 12:53:00,4,3,0,0,True
1,10,6,2020-08-08 02:39:00,2024-05-12 11:21:00,5,6,0,0,True
2,100,2,2022-12-19 07:25:00,2024-11-04 09:51:00,1,2,0,0,True
3,1000,3,2020-07-21 10:04:00,2023-08-10 15:09:00,3,2,0,0,True
4,1001,5,2020-01-12 04:03:00,2024-12-27 00:00:00,5,4,0,0,True


## 10. Construction de la table analytique

Cette table est la base finale pour Tableau.

Elle combine :

- leads ;
- contacts ;
- campagnes ;
- activités marketing agrégées.

In [10]:
# ============================================================
# 10. Jointures et table analytique
# ============================================================

crm = leads_clean.copy()

# --- Enrichissement contact via account_id -------------------------------
# Les leads ne possedent pas de contact_id direct. Le seul lien fiable est
# converted_account_id (lead) -> account_id (contacts). Seuls les leads
# convertis peuvent donc recuperer un telephone/contact : c'est une realite
# metier (le telephone n'apparait qu'apres conversion en compte), pas un bug.
CONTACT_COLS = {
    "contact_id": pd.NA, "contact_first_name": pd.NA, "contact_last_name": pd.NA,
    "contact_company": pd.NA, "contact_email_clean": pd.NA, "contact_email_valid": False,
    "contact_phone_clean": pd.NA, "contact_phone_valid": False,
    "contact_phone_issue": pd.NA, "contact_phone_extension": pd.NA,
    "contact_name_key": "", "contact_role": pd.NA,
}

if not contacts_clean.empty and "account_id" in contacts_clean.columns and "account_id" in crm.columns:
    cba = contacts_clean.dropna(subset=["account_id"]).copy()
    # Pour chaque compte, on retient le meilleur contact :
    # priorite au telephone valide, puis a l'email valide.
    cba["_rank"] = cba["contact_phone_valid"].astype(int) * 2 + cba["contact_email_valid"].astype(int)
    cba = cba.sort_values(["account_id", "_rank"], ascending=[True, False])
    keep = [c for c in CONTACT_COLS if c in cba.columns]
    contacts_by_account = cba.groupby("account_id", as_index=False).first()[["account_id"] + keep]
    crm = crm.merge(contacts_by_account, on="account_id", how="left")
    for col, default in CONTACT_COLS.items():
        if col not in crm.columns:
            crm[col] = default
else:
    for col, default in CONTACT_COLS.items():
        crm[col] = default

# Les booleens issus du merge sont NaN pour les leads sans contact rattache.
# Sans cette normalisation, bool(NaN) vaut True et fausse le choix du telephone.
for col in ["contact_phone_valid", "contact_email_valid"]:
    if col in crm.columns:
        crm[col] = crm[col].fillna(False).astype(bool)

# Email final : priorite au contact, sinon lead
crm["email_clean"] = crm["contact_email_clean"].combine_first(crm["lead_email_clean"])
crm["email_valid"] = crm["email_clean"].map(is_valid_email)

# Telephone final : priorite au numero valide du contact, sinon au numero valide du lead.
def choose_best_phone(row):
    for prefix in ["contact", "lead"]:
        valid_col = f"{prefix}_phone_valid"
        clean_col = f"{prefix}_phone_clean"
        ext_col = f"{prefix}_phone_extension"

        if valid_col in row.index and bool(row.get(valid_col, False)) and pd.notna(row.get(clean_col)):
            return pd.Series({
                "phone_clean": row.get(clean_col),
                "phone_valid": True,
                "phone_issue": "OK",
                "phone_source": prefix,
                "phone_extension": row.get(ext_col, pd.NA)
            })

    # Aucun numero valide : on garde la meilleure explication d'erreur disponible.
    for prefix in ["contact", "lead"]:
        issue_col = f"{prefix}_phone_issue"
        if issue_col in row.index and pd.notna(row.get(issue_col)):
            return pd.Series({
                "phone_clean": pd.NA,
                "phone_valid": False,
                "phone_issue": row.get(issue_col),
                "phone_source": prefix,
                "phone_extension": row.get(f"{prefix}_phone_extension", pd.NA)
            })

    return pd.Series({
        "phone_clean": pd.NA,
        "phone_valid": False,
        "phone_issue": "Telephone manquant",
        "phone_source": pd.NA,
        "phone_extension": pd.NA
    })

phone_final = crm.apply(choose_best_phone, axis=1)
crm = pd.concat([crm, phone_final], axis=1)

crm["company_clean"] = crm["contact_company"].combine_first(crm["lead_company"])
crm["company_key"] = crm["company_clean"].map(normalize_text_key)

# Jointure activites agregees
if not activities_by_lead.empty:
    crm = crm.merge(activities_by_lead, on="lead_id", how="left")
else:
    crm["total_activities"] = 0
    crm["first_activity_date"] = pd.NaT
    crm["last_activity_date"] = pd.NaT
    crm["unique_campaigns_count"] = 0
    crm["activity_types_count"] = 0
    crm["invalid_activity_dates"] = 0
    crm["future_activity_dates"] = 0
    crm["has_activity"] = False

activity_fill_cols = {
    "total_activities": 0,
    "unique_campaigns_count": 0,
    "activity_types_count": 0,
    "invalid_activity_dates": 0,
    "future_activity_dates": 0,
    "has_activity": False
}
for col, val in activity_fill_cols.items():
    if col in crm.columns:
        crm[col] = crm[col].fillna(val)

# Campagne principale :
# - si le lead a une campaign_id directe, on l'utilise ;
# - sinon, on recupere la campagne la plus recente depuis les activites.
latest_campaign_by_lead = pd.DataFrame()
if not activities_clean.empty and "campaign_id" in activities_clean.columns:
    latest_campaign_by_lead = (
        activities_clean
        .sort_values(["lead_id", "activity_date"])
        .dropna(subset=["campaign_id"])
        .groupby("lead_id")
        .tail(1)[["lead_id", "campaign_id"]]
        .rename(columns={"campaign_id": "activity_campaign_id"})
    )

if not latest_campaign_by_lead.empty:
    crm = crm.merge(latest_campaign_by_lead, on="lead_id", how="left")
else:
    crm["activity_campaign_id"] = pd.NA

crm["campaign_id"] = (
    crm["lead_campaign_id"].astype("string")
    .combine_first(crm["activity_campaign_id"].astype("string"))
)

# Jointure campaigns
if not campaigns_clean.empty and "campaign_id" in campaigns_clean.columns:
    crm = crm.merge(campaigns_clean, on="campaign_id", how="left")
else:
    crm["campaign_name"] = pd.NA
    crm["campaign_channel"] = pd.NA
    crm["campaign_source"] = pd.NA
    crm["campaign_start_date"] = pd.NaT
    crm["campaign_end_date"] = pd.NaT

# Source finale : source lead puis source campagne puis canal campagne
crm["source_final"] = crm["source_clean"].combine_first(crm["campaign_source"]).combine_first(crm["campaign_channel"])

display(crm.head())
print("Shape table analytique :", crm.shape)
print("Leads enrichis d'un contact (via account_id) :", crm["contact_id"].notna().sum())

print("Repartition des erreurs telephone finales :")
display(crm["phone_issue"].value_counts(dropna=False).rename_axis("phone_issue").reset_index(name="count").head(20))


,lead_id,contact_id_x,account_id,lead_campaign_id,lead_source_raw,source_clean,lead_status_raw,lead_status_clean,lead_created_at,lead_converted_at,lead_email_raw,lead_email_clean,lead_email_valid,lead_phone_raw,lead_phone_clean,lead_phone_digits,lead_phone_valid,lead_phone_issue,lead_phone_extension,lead_phone_clean_display,lead_company,lead_company_key,lead_first_name,lead_last_name,lead_name_key,lead_owner,contact_id_y,contact_first_name,contact_last_name,contact_company,contact_email_clean,contact_email_valid,contact_phone_clean,contact_phone_valid,contact_phone_issue,contact_phone_extension,contact_name_key,contact_role,contact_id,email_clean,email_valid,phone_clean,phone_valid,phone_issue,phone_source,phone_extension,company_clean,company_key,total_activities,first_activity_date,last_activity_date,unique_campaigns_count,activity_types_count,invalid_activity_dates,future_activity_dates,has_activity,activity_campaign_id,campaign_id,campaign_name,campaign_channel,campaign_source,campaign_start_date,campaign_end_date,source_final
0,1,<NA>,NaN,<NA>,Social Media,social media,MQL,mql,2021-05-27,NaT,sarah.martinez1@insightplatforms.com,sarah.martinez1@insightplatforms.com,True,<NA>,<NA>,<NA>,False,Téléphone manquant,<NA>,<NA>,Insight Platforms,insight platforms,Sarah,Martinez,sarah martinez,Non renseigne,NaN,NaN,NaN,NaN,NaN,False,NaN,False,NaN,NaN,NaN,NaN,<NA>,sarah.martinez1@insightplatforms.com,True,NaN,False,Téléphone manquant,lead,<NA>,Insight Platforms,insight platforms,4.0,2020-12-04 21:08:00,2024-05-09 12:53:00,4.0,3.0,0.0,0.0,True,128,128,Partner - Brand Awareness - Q2 2024,partner,partner,2024-05-08,2024-08-22,social media
1,2,<NA>,NaN,<NA>,Product-Led,product led,Working,working,2020-01-17,NaT,sarah.garcia2@techflowinc.com,sarah.garcia2@techflowinc.com,True,<NA>,<NA>,<NA>,False,Téléphone manquant,<NA>,<NA>,TechFlow Inc,techflow inc,Sarah,Garcia,sarah garcia,Non renseigne,NaN,NaN,NaN,NaN,NaN,False,NaN,False,NaN,NaN,NaN,NaN,<NA>,sarah.garcia2@techflowinc.com,True,NaN,False,Téléphone manquant,lead,<NA>,TechFlow Inc,techflow inc,5.0,2020-12-06 05:53:00,2024-07-07 22:10:00,5.0,4.0,0.0,0.0,True,13,13,Demand Gen Campaign - Organic Search,organic search,organic search,2024-05-07,2024-08-09,product led
2,3,<NA>,303,<NA>,Event,event,Converted,converted,2024-09-23,2024-11-30,robert.lee3@agiledynamics.com,robert.lee3@agiledynamics.com,True,<NA>,<NA>,<NA>,False,Téléphone manquant,<NA>,<NA>,Agile Dynamics,agile dynamics,Robert,Lee,robert lee,Non renseigne,2662,James,Sanchez,None,james.sanchez@umbrella.co,True,+12572336619,True,OK,None,james sanchez,Economic Buyer,<NA>,james.sanchez@umbrella.co,True,+12572336619,True,OK,contact,None,Agile Dynamics,agile dynamics,3.0,2020-04-30 13:16:00,2024-06-12 17:09:00,3.0,2.0,0.0,0.0,True,81,81,Webinar - Nurture - Q2 2024,webinar,webinar,2024-06-04,2024-07-31,event
3,4,<NA>,NaN,<NA>,Outbound - Cold,outbound cold,MQL,mql,2022-05-29,NaT,daniel.rodriguez4@metasync.com,daniel.rodriguez4@metasync.com,True,<NA>,<NA>,<NA>,False,Téléphone manquant,<NA>,<NA>,MetaSync,metasync,Daniel,Rodriguez,daniel rodriguez,Non renseigne,NaN,NaN,NaN,NaN,NaN,False,NaN,False,NaN,NaN,NaN,NaN,<NA>,daniel.rodriguez4@metasync.com,True,NaN,False,Téléphone manquant,lead,<NA>,MetaSync,metasync,4.0,2021-02-27 08:01:00,2024-09-29 17:17:00,4.0,4.0,0.0,0.0,True,198,198,FY2024 Nurture - Paid Search,paid search,paid search,2024-09-06,2024-10-10,outbound cold
4,5,<NA>,NaN,<NA>,Inbound - Webinar,inbound webinar,SQL,sql,2024-07-27,NaT,karen.lee5@scaleupsystems.com,karen.lee5@scaleupsystems.com,True,<NA>,<NA>,<NA>,False,Téléphone manquant,<NA>,<NA>,ScaleUp Systems,scaleup systems,Karen,Lee,karen lee,Non renseigne,NaN,NaN,NaN,NaN,NaN,False,NaN,False,NaN,NaN,NaN,NaN,<NA>,karen.lee5@scaleupsystems.com,True,NaN,False,Téléphone manquant,lead,<NA>,ScaleUp Systems,scaleup systems,3.0,2023-09-14 01:18:00,2024-10-02 21:16:00,3.0,2.0,0.0,0.0,True,198,198,FY2024 Nurture - Paid Search,paid search,paid search,2024-09-06,2024-10-10,inbound webinar


Shape table analytique : (8000, 64)
Leads enrichis d'un contact (via account_id) : 0
Repartition des erreurs telephone finales :


,phone_issue,count
0,Téléphone manquant,6581
1,OK,1134
2,Format possible mais numéro non validé,285


## 11. Contrôles qualité

On crée des indicateurs de qualité exploitables :

- complétude ;
- validité email / téléphone ;
- cohérence relationnelle ;
- dates futures ;
- doublons ;
- campagne associée ;
- activité marketing associée.

In [11]:
# ============================================================
# 11. Flags qualite
# ============================================================

today = pd.Timestamp.today().normalize()

crm["has_contact"] = crm["contact_id"].notna() & crm["contact_id"].astype(str).ne("<NA>") & crm["email_clean"].notna()
crm["has_campaign"] = crm["campaign_id"].notna() & crm["campaign_id"].astype(str).ne("<NA>")
crm["has_source"] = crm["source_final"].notna() & crm["source_final"].astype(str).str.strip().ne("")
crm["has_company"] = crm["company_clean"].notna() & crm["company_clean"].astype(str).str.strip().ne("")

crm["lead_created_valid"] = crm["lead_created_at"].notna()
crm["lead_created_future"] = crm["lead_created_at"].dt.tz_localize(None) > today

# --- Detection de doublons -------------------------------------------------
# Important : on dedoublonne sur l'EMAIL PROPRE AU LEAD (lead_email_clean),
# et non sur l'email consolide email_clean. En effet, l'enrichissement attache
# l'email/le telephone du contact du compte a TOUS les leads de ce compte ;
# ces champs sont donc partages par construction et ne doivent pas servir de
# signal de doublon (sinon plusieurs leads distincts d'un meme compte seraient
# faussement marques comme doublons).
crm["duplicate_email"] = False
crm["duplicate_phone"] = False

lead_email = crm.get("lead_email_clean", pd.Series(pd.NA, index=crm.index))
lead_email_ok = lead_email.map(is_valid_email) & lead_email.notna()
crm.loc[lead_email_ok, "duplicate_email"] = crm.loc[lead_email_ok, "lead_email_clean"].duplicated(keep=False)

# Les leads ne portent pas de telephone propre (le telephone provient du compte
# apres conversion) : le telephone n'est donc pas un signal de doublon de lead.

# Doublon "personne" probable : meme nom + meme entreprise, uniquement quand le
# nom est reellement renseigne. Fourni a titre indicatif (cf. note ci-dessous).
contact_nk = crm.get("contact_name_key", pd.Series("", index=crm.index)).fillna("").astype(str)
lead_nk = crm.get("lead_name_key", pd.Series("", index=crm.index)).fillna("").astype(str)
name_key = contact_nk.where(contact_nk.str.len() > 0, lead_nk)
company_key = crm["company_key"].fillna("").astype(str)

crm["duplicate_company_name"] = False
person_key = name_key + "|" + company_key
mask_person = (name_key.str.len() > 0) & (company_key.str.len() > 0)
crm.loc[mask_person, "duplicate_company_name"] = person_key[mask_person].duplicated(keep=False)

# Doublon retenu pour le pilotage = signal FIABLE : meme email de lead saisi
# plusieurs fois (vrais enregistrements en double dans le CRM).
# Le couple nom+entreprise n'est PAS inclus : le jeu synthetique ne compte
# qu'une vingtaine d'entreprises, ce qui rendrait le signal ininterpretable.
crm["is_duplicate"] = crm["duplicate_email"] | crm["duplicate_phone"]

# Champs manquants majeurs
crm["missing_critical_fields_count"] = (
    (~crm["email_valid"]).astype(int)
    + (~crm["phone_valid"]).astype(int)
    + (~crm["has_source"]).astype(int)
    + (~crm["has_company"]).astype(int)
    + (~crm["has_campaign"]).astype(int)
)

quality_flags = [
    "email_valid", "phone_valid", "has_contact", "has_campaign", "has_source",
    "has_company", "has_activity", "lead_created_valid", "lead_created_future",
    "duplicate_email", "duplicate_phone", "duplicate_company_name", "is_duplicate"
]

display(crm[quality_flags].mean(numeric_only=True).mul(100).round(2).rename("pct_true").reset_index())


,index,pct_true
0,email_valid,93.79
1,phone_valid,14.18
2,has_contact,0.00
3,has_campaign,97.05
4,has_source,99.88
5,has_company,100.00
6,lead_created_valid,100.00
7,lead_created_future,0.00
8,duplicate_email,0.00
9,duplicate_phone,0.00


## 12. Score qualité CRM

Score proposé sur 100 :

| Critère | Points |
|---|---:|
| Email valide | 25 |
| Téléphone valide | 15 |
| Contact exploitable | 10 |
| Source marketing renseignée | 10 |
| Entreprise renseignée | 10 |
| Campagne ou activité associée | 10 |
| Date de création valide et non future | 10 |
| Pas de doublon | 10 |

Le score permet de passer d’un diagnostic technique à un indicateur métier.

In [12]:
# ============================================================
# 12. Score qualité et statut campagne
# ============================================================

crm["date_quality_ok"] = crm["lead_created_valid"] & (~crm["lead_created_future"])

crm["quality_score"] = (
    25 * crm["email_valid"].astype(int)
    + 15 * crm["phone_valid"].astype(int)
    + 10 * crm["has_contact"].astype(int)
    + 10 * crm["has_source"].astype(int)
    + 10 * crm["has_company"].astype(int)
    + 10 * (crm["has_campaign"] | crm["has_activity"].astype(bool)).astype(int)
    + 10 * crm["date_quality_ok"].astype(int)
    + 10 * (~crm["is_duplicate"]).astype(int)
)

crm["quality_level"] = crm["quality_score"].apply(quality_level)

crm["campaign_ready"] = (
    crm["email_valid"]
    & (~crm["is_duplicate"])
    & crm["has_source"]
    & (crm["quality_score"] >= 60)
)

crm["campaign_ready_label"] = yes_no(crm["campaign_ready"])

crm["cleaning_priority"] = np.select(
    [
        crm["quality_score"] < 40,
        (crm["quality_score"] < 60) | crm["is_duplicate"] | (~crm["email_valid"]),
        (crm["quality_score"] < 80) | (~crm["phone_valid"])
    ],
    [
        "Priorité haute",
        "Priorité moyenne",
        "Priorité faible"
    ],
    default="OK"
)

display(
    crm[["lead_id", "email_clean", "phone_clean", "source_final", "campaign_name",
         "quality_score", "quality_level", "campaign_ready_label", "cleaning_priority"]].head(10)
)

display(crm["quality_level"].value_counts(dropna=False).rename_axis("quality_level").reset_index(name="leads_count"))

,lead_id,email_clean,phone_clean,source_final,campaign_name,quality_score,quality_level,campaign_ready_label,cleaning_priority
0,1,sarah.martinez1@insightplatforms.com,NaN,social media,Partner - Brand Awareness - Q2 2024,75,Correct,Oui,Priorité faible
1,2,sarah.garcia2@techflowinc.com,NaN,product led,Demand Gen Campaign - Organic Search,75,Correct,Oui,Priorité faible
2,3,james.sanchez@umbrella.co,+12572336619,event,Webinar - Nurture - Q2 2024,90,Excellent,Oui,OK
3,4,daniel.rodriguez4@metasync.com,NaN,outbound cold,FY2024 Nurture - Paid Search,75,Correct,Oui,Priorité faible
4,5,karen.lee5@scaleupsystems.com,NaN,inbound webinar,FY2024 Nurture - Paid Search,75,Correct,Oui,Priorité faible
5,6,james.wilson6@cloudscalesystems.com,NaN,inbound content,FY2022 ABM - Referral,75,Correct,Oui,Priorité faible
6,7,robert.rodriguez7@vertexai.com,NaN,social media,Content Syndication - Event Promotion - Q2 2024,75,Correct,Oui,Priorité faible
7,8,lisa.jones8@brightwavetech.com,NaN,event,FY2023 Product Launch - Direct Mail,75,Correct,Oui,Priorité faible
8,9,james.garcia9@agiledynamics.com,NaN,inbound content,FY2023 Brand Awareness - Organic Search,75,Correct,Oui,Priorité faible
9,10,william.taylor10@agiledynamics.com,NaN,outbound cold,Partner - Brand Awareness - Q2 2024,75,Correct,Oui,Priorité faible


,quality_level,leads_count
0,Correct,6421
1,Excellent,1103
2,À corriger,476


## 13. Journal des problèmes qualité

Cette table est très utile dans Tableau.

Elle permet de construire une vue **actions correctives** :

- type d’erreur ;
- champ concerné ;
- sévérité ;
- action recommandée.

In [13]:
# ============================================================
# 13. Création du journal des problèmes qualité
# ============================================================

def add_issue(issues, condition, issue_type, field_name, severity, recommended_action, detail_col=None):
    tmp = crm.loc[condition, ["lead_id"]].copy()
    if tmp.empty:
        return issues

    tmp["record_type"] = "lead"
    tmp["issue_type"] = issue_type
    tmp["field_name"] = field_name
    tmp["severity"] = severity
    tmp["recommended_action"] = recommended_action

    if detail_col and detail_col in crm.columns:
        tmp["issue_detail"] = crm.loc[condition, detail_col].astype("string").values
    else:
        tmp["issue_detail"] = pd.NA

    return issues + [tmp]

issues = []

issues = add_issue(
    issues,
    ~crm["email_valid"],
    "Email invalide ou manquant",
    "email",
    "Haute",
    "Corriger l'email ou exclure le lead des campagnes email."
)

issues = add_issue(
    issues,
    ~crm["phone_valid"],
    "Téléphone invalide ou manquant",
    "phone",
    "Moyenne",
    "Corriger au format international E.164, par exemple +14155552671, ou enrichir avant prospection.",
    detail_col="phone_issue"
)

issues = add_issue(
    issues,
    ~crm["has_source"],
    "Source marketing manquante",
    "source",
    "Moyenne",
    "Renseigner la source pour fiabiliser l'attribution marketing."
)

issues = add_issue(
    issues,
    ~crm["has_company"],
    "Entreprise manquante",
    "company",
    "Moyenne",
    "Enrichir l'entreprise pour améliorer la segmentation B2B."
)

issues = add_issue(
    issues,
    ~crm["has_campaign"],
    "Campagne non associée",
    "campaign_id",
    "Faible",
    "Contrôler l'attribution campagne ou rattacher le lead à une campagne."
)

issues = add_issue(
    issues,
    crm["lead_created_future"],
    "Date future incohérente",
    "lead_created_at",
    "Haute",
    "Corriger la date ou exclure la ligne des analyses temporelles."
)

issues = add_issue(
    issues,
    crm["duplicate_email"],
    "Doublon email",
    "email",
    "Haute",
    "Fusionner les fiches ou conserver la fiche la plus complète."
)

issues = add_issue(
    issues,
    crm["duplicate_phone"],
    "Doublon téléphone",
    "phone",
    "Moyenne",
    "Vérifier si plusieurs leads représentent le même contact."
)

issues = add_issue(
    issues,
    crm["duplicate_company_name"],
    "Doublon probable nom + entreprise",
    "company/contact_name",
    "Moyenne",
    "Comparer les fiches et fusionner si nécessaire."
)

crm_quality_issues = pd.concat(issues, ignore_index=True) if issues else pd.DataFrame(
    columns=[
        "lead_id", "record_type", "issue_type", "field_name",
        "severity", "recommended_action", "issue_detail"
    ]
)

display(crm_quality_issues.head(20))
display(crm_quality_issues["issue_type"].value_counts().rename_axis("issue_type").reset_index(name="count"))

print("Détail des problèmes de téléphone :")
display(
    crm_quality_issues
    .query("field_name == 'phone'")
    ["issue_detail"]
    .value_counts(dropna=False)
    .rename_axis("issue_detail")
    .reset_index(name="count")
)

,lead_id,record_type,issue_type,field_name,severity,recommended_action,issue_detail
0,25,lead,Email invalide ou manquant,email,Haute,Corriger l'email ou exclure le lead des campag...,<NA>
1,56,lead,Email invalide ou manquant,email,Haute,Corriger l'email ou exclure le lead des campag...,<NA>
2,57,lead,Email invalide ou manquant,email,Haute,Corriger l'email ou exclure le lead des campag...,<NA>
3,60,lead,Email invalide ou manquant,email,Haute,Corriger l'email ou exclure le lead des campag...,<NA>
4,96,lead,Email invalide ou manquant,email,Haute,Corriger l'email ou exclure le lead des campag...,<NA>
5,114,lead,Email invalide ou manquant,email,Haute,Corriger l'email ou exclure le lead des campag...,<NA>
6,117,lead,Email invalide ou manquant,email,Haute,Corriger l'email ou exclure le lead des campag...,<NA>
7,120,lead,Email invalide ou manquant,email,Haute,Corriger l'email ou exclure le lead des campag...,<NA>
8,126,lead,Email invalide ou manquant,email,Haute,Corriger l'email ou exclure le lead des campag...,<NA>
9,157,lead,Email invalide ou manquant,email,Haute,Corriger l'email ou exclure le lead des campag...,<NA>


,issue_type,count
0,Téléphone invalide ou manquant,6866
1,Doublon probable nom + entreprise,3671
2,Email invalide ou manquant,497
3,Campagne non associée,236
4,Source marketing manquante,10


Détail des problèmes de téléphone :


,issue_detail,count
0,Téléphone manquant,6581
1,Format possible mais numéro non validé,285


## 14. Analyses métier pour le dashboard

Ces analyses servent à orienter la construction Tableau :

- vue direction ;
- vue par source marketing ;
- vue par campagne ;
- vue actions correctives.

In [14]:
# ============================================================
# 14. KPIs et analyses pour Tableau
# ============================================================

kpi_summary = pd.DataFrame([{
    "total_leads": len(crm),
    "quality_score_avg": round(crm["quality_score"].mean(), 2),
    "campaign_ready_rate_pct": round(crm["campaign_ready"].mean() * 100, 2),
    "email_valid_rate_pct": round(crm["email_valid"].mean() * 100, 2),
    "phone_valid_rate_pct": round(crm["phone_valid"].mean() * 100, 2),
    "duplicate_rate_pct": round(crm["is_duplicate"].mean() * 100, 2),
    "critical_leads": int((crm["quality_level"] == "Critique").sum()),
    "issues_total": len(crm_quality_issues)
}])

display(kpi_summary)

quality_by_source = (
    crm
    .groupby("source_final", dropna=False)
    .agg(
        leads_count=("lead_id", "count"),
        quality_score_avg=("quality_score", "mean"),
        campaign_ready_rate=("campaign_ready", "mean"),
        email_valid_rate=("email_valid", "mean"),
        duplicate_rate=("is_duplicate", "mean")
    )
    .reset_index()
)

quality_by_source["quality_score_avg"] = quality_by_source["quality_score_avg"].round(2)

for col in ["campaign_ready_rate", "email_valid_rate", "duplicate_rate"]:
    quality_by_source[col] = (quality_by_source[col] * 100).round(2)

display(quality_by_source.sort_values("quality_score_avg", ascending=False).head(15))

quality_by_campaign = (
    crm
    .groupby(["campaign_id", "campaign_name"], dropna=False)
    .agg(
        leads_count=("lead_id", "count"),
        quality_score_avg=("quality_score", "mean"),
        campaign_ready_rate=("campaign_ready", "mean"),
        total_activities=("total_activities", "sum"),
        duplicate_rate=("is_duplicate", "mean")
    )
    .reset_index()
)

quality_by_campaign["quality_score_avg"] = quality_by_campaign["quality_score_avg"].round(2)
quality_by_campaign["campaign_ready_rate"] = (quality_by_campaign["campaign_ready_rate"] * 100).round(2)
quality_by_campaign["duplicate_rate"] = (quality_by_campaign["duplicate_rate"] * 100).round(2)

display(quality_by_campaign.sort_values(["leads_count", "quality_score_avg"], ascending=[False, False]).head(15))

,total_leads,quality_score_avg,campaign_ready_rate_pct,email_valid_rate_pct,phone_valid_rate_pct,duplicate_rate_pct,critical_leads,issues_total
0,8000,75.31,93.66,93.79,14.18,0.0,0,11280


,source_final,leads_count,quality_score_avg,campaign_ready_rate,email_valid_rate,duplicate_rate
17,webinar,29,77.76,96.55,96.55,0.0
7,organic search,18,77.50,100.00,100.00,0.0
13,partner referral,745,76.84,94.90,94.90,0.0
14,product led,774,76.24,93.15,93.15,0.0
6,inbound webinar,769,76.07,94.93,94.93,0.0
3,event,783,75.90,95.02,95.02,0.0
5,inbound web,813,75.50,93.73,93.73,0.0
10,paid search,785,75.39,93.12,93.12,0.0
2,email,27,75.37,92.59,92.59,0.0
11,paid social,19,75.26,94.74,94.74,0.0


,campaign_id,campaign_name,leads_count,quality_score_avg,campaign_ready_rate,total_activities,duplicate_rate
200,<NA>,NaN,236,67.67,90.68,38.0,0.0
193,93,Event - Product Launch - Q4 2024,134,75.71,94.78,615.0,0.0
74,166,FY2024 Demand Gen - Organic Search,134,75.04,90.30,625.0,0.0
21,118,FY2024 Brand Awareness - Referral,132,75.49,94.70,586.0,0.0
163,66,Webinar - Thought Leadership - Q4 2024,130,76.08,94.62,619.0,0.0
101,190,Event - Event Promotion - Q4 2024,123,75.77,94.31,533.0,0.0
117,24,FY2024 Brand Awareness - Content Syndication,122,74.02,90.16,590.0,0.0
162,65,Product Launch Campaign - Paid Social,120,76.12,95.00,546.0,0.0
96,186,Webinar - Demand Gen - Q4 2024,118,75.64,92.37,524.0,0.0
131,37,Product Launch Campaign - Email,117,76.28,94.87,504.0,0.0


## 15. Exports pour Tableau

Fichiers produits :

- `crm_quality_clean.csv` : table principale à connecter dans Tableau ;
- `crm_quality_issues.csv` : table des problèmes pour la page “actions correctives” ;
- `data_dictionary.csv` : documentation des colonnes ;
- `kpi_summary.csv` : synthèse des KPIs.

In [15]:
# ============================================================
# 15. Export des fichiers pour Tableau
# ============================================================

preferred_cols = [
    "lead_id", "contact_id", "campaign_id",
    "lead_owner",
    "email_clean", "phone_clean", "phone_valid", "phone_issue", "phone_source", "phone_extension",
    "company_clean",
    "source_final", "lead_status_clean",
    "campaign_name", "campaign_channel", "campaign_source",
    "lead_created_at", "lead_converted_at",
    "campaign_start_date", "campaign_end_date",
    "first_activity_date", "last_activity_date",
    "total_activities", "unique_campaigns_count", "activity_types_count",
    "email_valid", "has_contact", "has_campaign",
    "has_source", "has_company", "has_activity",
    "lead_created_valid", "lead_created_future",
    "duplicate_email", "duplicate_phone", "duplicate_company_name", "is_duplicate",
    "missing_critical_fields_count",
    "quality_score", "quality_level",
    "campaign_ready", "campaign_ready_label",
    "cleaning_priority"
]

final_cols = [c for c in preferred_cols if c in crm.columns]
crm_quality_clean = crm[final_cols].copy()

# Nettoyage final des dates pour export CSV
for col in crm_quality_clean.columns:
    if pd.api.types.is_datetime64_any_dtype(crm_quality_clean[col]):
        crm_quality_clean[col] = crm_quality_clean[col].dt.strftime("%Y-%m-%d %H:%M:%S")

clean_path = PROCESSED_DIR / "crm_quality_clean.csv"
issues_path = PROCESSED_DIR / "crm_quality_issues.csv"
kpi_path = REPORTS_DIR / "kpi_summary.csv"

crm_quality_clean.to_csv(clean_path, index=False, encoding="utf-8")
crm_quality_issues.to_csv(issues_path, index=False, encoding="utf-8")
kpi_summary.to_csv(kpi_path, index=False, encoding="utf-8")

data_dictionary = pd.DataFrame([
    {"column": "lead_id", "description": "Identifiant unique du lead."},
    {"column": "contact_id", "description": "Identifiant du contact rattaché au lead."},
    {"column": "campaign_id", "description": "Identifiant de la campagne marketing associée."},
    {"column": "email_clean", "description": "Email normalisé."},
    {"column": "phone_clean", "description": "Téléphone propre au format international E.164 lorsque le numéro est valide."},
    {"column": "phone_valid", "description": "Indique si le téléphone est exploitable."},
    {"column": "phone_issue", "description": "Explication de l'erreur téléphone : trop court, trop long, manquant, indicatif ambigu, etc."},
    {"column": "phone_source", "description": "Origine du téléphone retenu : contact ou lead."},
    {"column": "source_final", "description": "Source marketing consolidée."},
    {"column": "quality_score", "description": "Score qualité CRM sur 100."},
    {"column": "quality_level", "description": "Niveau qualité : Excellent, Correct, À corriger, Critique."},
    {"column": "campaign_ready", "description": "Indique si le lead est exploitable pour une campagne marketing."},
    {"column": "cleaning_priority", "description": "Priorité de correction opérationnelle."},
    {"column": "is_duplicate", "description": "Indique si le lead est un doublon exact ou probable."},
])

dict_path = PROCESSED_DIR / "data_dictionary.csv"
data_dictionary.to_csv(dict_path, index=False, encoding="utf-8")

print("Exports générés :")
print("-", clean_path.resolve())
print("-", issues_path.resolve())
print("-", dict_path.resolve())
print("-", kpi_path.resolve())

display(crm_quality_clean.head())

Exports générés :
- C:\Users\user\Desktop\Study\M2\Data visualisations\Projet final\data\processed\crm_quality_clean.csv
- C:\Users\user\Desktop\Study\M2\Data visualisations\Projet final\data\processed\crm_quality_issues.csv
- C:\Users\user\Desktop\Study\M2\Data visualisations\Projet final\data\processed\data_dictionary.csv
- C:\Users\user\Desktop\Study\M2\Data visualisations\Projet final\reports\kpi_summary.csv


,lead_id,contact_id,campaign_id,lead_owner,email_clean,phone_clean,phone_valid,phone_issue,phone_source,phone_extension,company_clean,source_final,lead_status_clean,campaign_name,campaign_channel,campaign_source,lead_created_at,lead_converted_at,campaign_start_date,campaign_end_date,first_activity_date,last_activity_date,total_activities,unique_campaigns_count,activity_types_count,email_valid,has_contact,has_campaign,has_source,has_company,has_activity,lead_created_valid,lead_created_future,duplicate_email,duplicate_phone,duplicate_company_name,is_duplicate,missing_critical_fields_count,quality_score,quality_level,campaign_ready,campaign_ready_label,cleaning_priority
0,1,<NA>,128,Non renseigne,sarah.martinez1@insightplatforms.com,NaN,False,Téléphone manquant,lead,<NA>,Insight Platforms,social media,mql,Partner - Brand Awareness - Q2 2024,partner,partner,2021-05-27 00:00:00,NaN,2024-05-08 00:00:00,2024-08-22 00:00:00,2020-12-04 21:08:00,2024-05-09 12:53:00,4.0,4.0,3.0,True,False,True,True,True,True,True,False,False,False,True,False,1,75,Correct,True,Oui,Priorité faible
1,2,<NA>,13,Non renseigne,sarah.garcia2@techflowinc.com,NaN,False,Téléphone manquant,lead,<NA>,TechFlow Inc,product led,working,Demand Gen Campaign - Organic Search,organic search,organic search,2020-01-17 00:00:00,NaN,2024-05-07 00:00:00,2024-08-09 00:00:00,2020-12-06 05:53:00,2024-07-07 22:10:00,5.0,5.0,4.0,True,False,True,True,True,True,True,False,False,False,True,False,1,75,Correct,True,Oui,Priorité faible
2,3,<NA>,81,Non renseigne,james.sanchez@umbrella.co,+12572336619,True,OK,contact,None,Agile Dynamics,event,converted,Webinar - Nurture - Q2 2024,webinar,webinar,2024-09-23 00:00:00,2024-11-30 00:00:00,2024-06-04 00:00:00,2024-07-31 00:00:00,2020-04-30 13:16:00,2024-06-12 17:09:00,3.0,3.0,2.0,True,False,True,True,True,True,True,False,False,False,False,False,0,90,Excellent,True,Oui,OK
3,4,<NA>,198,Non renseigne,daniel.rodriguez4@metasync.com,NaN,False,Téléphone manquant,lead,<NA>,MetaSync,outbound cold,mql,FY2024 Nurture - Paid Search,paid search,paid search,2022-05-29 00:00:00,NaN,2024-09-06 00:00:00,2024-10-10 00:00:00,2021-02-27 08:01:00,2024-09-29 17:17:00,4.0,4.0,4.0,True,False,True,True,True,True,True,False,False,False,True,False,1,75,Correct,True,Oui,Priorité faible
4,5,<NA>,198,Non renseigne,karen.lee5@scaleupsystems.com,NaN,False,Téléphone manquant,lead,<NA>,ScaleUp Systems,inbound webinar,sql,FY2024 Nurture - Paid Search,paid search,paid search,2024-07-27 00:00:00,NaN,2024-09-06 00:00:00,2024-10-10 00:00:00,2023-09-14 01:18:00,2024-10-02 21:16:00,3.0,3.0,2.0,True,False,True,True,True,True,True,False,False,False,True,False,1,75,Correct,True,Oui,Priorité faible


## 16. Recommandations Tableau

### Page 1 — Vue Direction

**KPIs à afficher :**

- Score qualité moyen ;
- Nombre total de leads ;
- % leads prêts pour campagne ;
- % emails valides ;
- % téléphones valides ;
- nombre de doublons ;
- nombre de leads critiques.

**Graphiques :**

- répartition des `quality_level` ;
- évolution de la qualité par date de création ;
- top sources par score qualité.

---

### Page 2 — Qualité par source marketing

**Objectif :** identifier les sources qui produisent les leads les plus fiables.

**Graphiques :**

- `quality_score` moyen par `source_final` ;
- taux de `campaign_ready` par source ;
- taux de doublons par source ;
- taux d’emails invalides par source ;
- taux de téléphones invalides par source.

**Décisions :**

- investir sur les sources fiables ;
- corriger les formulaires des sources faibles ;
- surveiller les sources générant beaucoup de doublons.

---

### Page 3 — Qualité par campagne

**Objectif :** comparer les campagnes marketing.

**Graphiques :**

- score qualité par campagne ;
- volume de leads par campagne ;
- taux de leads prêts pour campagne ;
- nombre d’activités par campagne.

**Décisions :**

- prioriser les campagnes à forte qualité ;
- améliorer les campagnes générant des leads incomplets ;
- fiabiliser l’attribution marketing.

---

### Page 4 — Actions correctives

À connecter avec `crm_quality_issues.csv`.

**Graphiques :**

- nombre de problèmes par type ;
- problèmes par sévérité ;
- détail des erreurs téléphone via `issue_detail` ;
- liste filtrable des leads à corriger ;
- actions recommandées.

**Décisions :**

- corriger les emails invalides avant emailing ;
- corriger les téléphones au format international `E.164` ;
- fusionner les doublons ;
- enrichir les leads sans téléphone / entreprise ;
- corriger les sources marketing manquantes.

In [16]:
# ============================================================
# 17. Contrôle final des fichiers générés
# ============================================================

for path in [clean_path, issues_path, dict_path, kpi_path]:
    print(path, "=>", "OK" if path.exists() else "ABSENT", "| taille :", path.stat().st_size if path.exists() else 0, "octets")

print("\nAperçu fichier propre :")
display(pd.read_csv(clean_path).head())

print("\nAperçu fichier issues :")
display(pd.read_csv(issues_path).head())

data\processed\crm_quality_clean.csv => OK | taille : 3192419 octets
data\processed\crm_quality_issues.csv => OK | taille : 1766145 octets
data\processed\data_dictionary.csv => OK | taille : 896 octets
reports\kpi_summary.csv => OK | taille : 186 octets

Aperçu fichier propre :


,lead_id,contact_id,campaign_id,lead_owner,email_clean,phone_clean,phone_valid,phone_issue,phone_source,phone_extension,company_clean,source_final,lead_status_clean,campaign_name,campaign_channel,campaign_source,lead_created_at,lead_converted_at,campaign_start_date,campaign_end_date,first_activity_date,last_activity_date,total_activities,unique_campaigns_count,activity_types_count,email_valid,has_contact,has_campaign,has_source,has_company,has_activity,lead_created_valid,lead_created_future,duplicate_email,duplicate_phone,duplicate_company_name,is_duplicate,missing_critical_fields_count,quality_score,quality_level,campaign_ready,campaign_ready_label,cleaning_priority
0,1,NaN,128.0,Non renseigne,sarah.martinez1@insightplatforms.com,NaN,False,Téléphone manquant,lead,NaN,Insight Platforms,social media,mql,Partner - Brand Awareness - Q2 2024,partner,partner,2021-05-27 00:00:00,NaN,2024-05-08 00:00:00,2024-08-22 00:00:00,2020-12-04 21:08:00,2024-05-09 12:53:00,4.0,4.0,3.0,True,False,True,True,True,True,True,False,False,False,True,False,1,75,Correct,True,Oui,Priorité faible
1,2,NaN,13.0,Non renseigne,sarah.garcia2@techflowinc.com,NaN,False,Téléphone manquant,lead,NaN,TechFlow Inc,product led,working,Demand Gen Campaign - Organic Search,organic search,organic search,2020-01-17 00:00:00,NaN,2024-05-07 00:00:00,2024-08-09 00:00:00,2020-12-06 05:53:00,2024-07-07 22:10:00,5.0,5.0,4.0,True,False,True,True,True,True,True,False,False,False,True,False,1,75,Correct,True,Oui,Priorité faible
2,3,NaN,81.0,Non renseigne,james.sanchez@umbrella.co,1.257234e+10,True,OK,contact,NaN,Agile Dynamics,event,converted,Webinar - Nurture - Q2 2024,webinar,webinar,2024-09-23 00:00:00,2024-11-30 00:00:00,2024-06-04 00:00:00,2024-07-31 00:00:00,2020-04-30 13:16:00,2024-06-12 17:09:00,3.0,3.0,2.0,True,False,True,True,True,True,True,False,False,False,False,False,0,90,Excellent,True,Oui,OK
3,4,NaN,198.0,Non renseigne,daniel.rodriguez4@metasync.com,NaN,False,Téléphone manquant,lead,NaN,MetaSync,outbound cold,mql,FY2024 Nurture - Paid Search,paid search,paid search,2022-05-29 00:00:00,NaN,2024-09-06 00:00:00,2024-10-10 00:00:00,2021-02-27 08:01:00,2024-09-29 17:17:00,4.0,4.0,4.0,True,False,True,True,True,True,True,False,False,False,True,False,1,75,Correct,True,Oui,Priorité faible
4,5,NaN,198.0,Non renseigne,karen.lee5@scaleupsystems.com,NaN,False,Téléphone manquant,lead,NaN,ScaleUp Systems,inbound webinar,sql,FY2024 Nurture - Paid Search,paid search,paid search,2024-07-27 00:00:00,NaN,2024-09-06 00:00:00,2024-10-10 00:00:00,2023-09-14 01:18:00,2024-10-02 21:16:00,3.0,3.0,2.0,True,False,True,True,True,True,True,False,False,False,True,False,1,75,Correct,True,Oui,Priorité faible



Aperçu fichier issues :


,lead_id,record_type,issue_type,field_name,severity,recommended_action,issue_detail
0,25,lead,Email invalide ou manquant,email,Haute,Corriger l'email ou exclure le lead des campag...,NaN
1,56,lead,Email invalide ou manquant,email,Haute,Corriger l'email ou exclure le lead des campag...,NaN
2,57,lead,Email invalide ou manquant,email,Haute,Corriger l'email ou exclure le lead des campag...,NaN
3,60,lead,Email invalide ou manquant,email,Haute,Corriger l'email ou exclure le lead des campag...,NaN
4,96,lead,Email invalide ou manquant,email,Haute,Corriger l'email ou exclure le lead des campag...,NaN


# Conclusion

Ce notebook transforme un bundle GTM relationnel brut en fichiers exploitables dans Tableau.

## Valeur métier

Le projet permet de répondre à des questions concrètes :

- quels leads peuvent être utilisés en campagne ;
- quelles sources produisent des données fiables ;
- quelles campagnes génèrent des leads incomplets ;
- quels problèmes doivent être corrigés en priorité ;
- quelle est la fiabilité globale du CRM.

## Livrables finaux

- Notebook Python : audit, nettoyage, scoring, exports.
- CSV Tableau : `crm_quality_clean.csv`.
- CSV actions correctives : `crm_quality_issues.csv`.
- Dashboard Tableau : pilotage qualité CRM et readiness marketing.